# Emotion Classification with roberta-base

This notebook demonstrates **supervised emotion classification** using **roberta-base**, a pretrained English transformer model from Hugging Face.

The model is fine-tuned on a labeled emotion dataset, where each sentence is mapped to a single emotion category.

### Problem Setup
- **Input**: Text sentence
- **Output**: Emotion label (e.g. joy, sadness, anger)
- **Learning type**: Supervised learning
- **Task**: Multi-class classification

### High-Level Pipeline
1. Load and inspect the dataset
2. Convert emotion labels into numerical IDs
3. Tokenize text into input IDs and attention masks
4. Train (fine-tune) roberta-base
5. Evaluate on validation and test sets
6. Run inference on new sentences

In [13]:
import numpy as np
import pandas as pd

## Step 1: Dataset and Multi-Label Emotion Setup

This project uses a **multi-label emotion dataset** for supervised learning.

Each text can express **multiple emotions at the same time**.
For example:
- A sentence can be both *sad* and *angry*

Dataset structure:
- `text`: the input sentence
- Multiple emotion columns (binary values)
  - `1` → emotion is present
  - `0` → emotion is not present

Because the correct emotions are known in advance, the model can:
- Compare predicted emotions against ground-truth labels
- Calculate loss
- Learn through backpropagation

In [14]:
# Load dataset
df = pd.read_csv("../goemotions_cleaned.csv")

# Identify emotion columns (all columns except text)
emotion_cols = df.columns[9:]

# Keep only text and emotion labels
df = df[["text"] + list(emotion_cols)]

In [15]:
print(df.columns)

Index(['text', 'admiration', 'amusement', 'anger', 'annoyance', 'approval',
       'caring', 'confusion', 'curiosity', 'desire', 'disappointment',
       'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear',
       'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride',
       'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='str')


## Step 2: Train / Validation / Test Split (70/20/10)

We split the dataset into three sets for training and evaluation:

- **Train set (70%)**: Used to fine-tune the model  
- **Validation set (20%)**: Used to monitor performance and adjust hyperparameters  
- **Test set (10%)**: Used for final evaluation on unseen data

Shuffling the dataset ensures randomness, and `random_state` guarantees reproducibility.

In [16]:
from sklearn.model_selection import train_test_split

# First, split into train (70%) and temp (30%)
df_train, df_temp = train_test_split(
    df, test_size=0.3, random_state=42, shuffle=True
)

# Split temp into validation (20%) and test (10%)
# temp is 30% of total, so:
# validation = 20/30 = 2/3 of temp
# test = 10/30 = 1/3 of temp
df_val, df_test = train_test_split(
    df_temp, test_size=1/3, random_state=42, shuffle=True
)

# Print dataset sizes
print(f"Train: {len(df_train)} rows")
print(f"Validation: {len(df_val)} rows")
print(f"Test: {len(df_test)} rows")

Train: 40602 rows
Validation: 11601 rows
Test: 5801 rows


## Step 3: Convert Multi-Label Emotions into Numeric Vectors

The model cannot work directly with text labels like `"joy"` or `"anger"`.  
We need to convert each emotion label into a **numeric vector**.

For multi-label classification:
- Each sentence has a **vector of 0s and 1s**, one position per emotion
  - `1` → emotion is present
  - `0` → emotion is absent

Example:
| text                     | joy | sadness | anger | fear |
|--------------------------|-----|---------|-------|-----|
| "I feel nervous"         | 0   | 0       | 0     | 1   |
| "I’m excited but anxious"| 1   | 0       | 0     | 1   |

These vectors become the **ground truth labels** for training.

In [17]:
# Get emotion column names
emotion_cols = df_train.columns[9:]

# Extract label vectors for train, validation, test sets
y_train = df_train[emotion_cols].values
y_val   = df_val[emotion_cols].values
y_test  = df_test[emotion_cols].values


## Step 4: Tokenization

Neural network models cannot work with raw text. We need to convert sentences into numbers.

Steps:

1. **Tokenizer** converts text into `input_ids`:
   - Each word (or subword) gets a unique number.
   - Example: "I feel happy" → `[101, 1045, 3115, 3793, 102]`  

2. **Attention mask**:
   - Tells the model which tokens to pay attention to.
   - `1` = real token, `0` = padding

3. **Labels**:
   - Already converted to numeric vectors (`y_train`, `y_val`, `y_test`)

In [18]:
from transformers import AutoTokenizer

# Load the main English RoBERTa tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# Convert each split into lists of sentences
train_texts = df_train['text'].tolist()
val_texts   = df_val['text'].tolist()
test_texts  = df_test['text'].tolist()

# Tokenize function
def tokenize_texts(texts):
    return tokenizer(
        texts,
        padding='max_length',   # pad all sentences to the same length
        truncation=True,        # cut sentences that are too long
        max_length=128,         # max tokens per sentence
        return_tensors='pt'     # return PyTorch tensors
    )

# Apply tokenization to all splits
train_tokens = tokenize_texts(train_texts)
val_tokens   = tokenize_texts(val_texts)
test_tokens  = tokenize_texts(test_texts)

## Step 5: Prepare Data for Training

- Our model can’t read text directly. It reads **numbers in tensors**.
- Each sentence has:
  - `input_ids` → numbers representing words
  - `attention_mask` → tells the model which numbers are real tokens vs padding
  - `labels` → multi-label vector of 0s and 1s

- We use a **Dataset** to bundle these together per sentence.
- We use a **DataLoader** to feed **batches of sentences** to the model.
  - Batch = multiple sentences at once (faster training)
  - Shuffle = randomizes sentences for better learning

In [19]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert labels to PyTorch tensors
y_train_tensor = torch.tensor(y_train, dtype=torch.float)
y_val_tensor   = torch.tensor(y_val, dtype=torch.float)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float)

# Create datasets (each dataset = inputs + labels)
train_dataset = TensorDataset(train_tokens['input_ids'], 
                              train_tokens['attention_mask'], 
                              y_train_tensor)

val_dataset   = TensorDataset(val_tokens['input_ids'], 
                              val_tokens['attention_mask'], 
                              y_val_tensor)

test_dataset  = TensorDataset(test_tokens['input_ids'], 
                              test_tokens['attention_mask'], 
                              y_test_tensor)

# DataLoaders: feed data in batches
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)


## Step 6: Train the Model

1. **What we’re doing:**
   - The model takes a batch of sentences (numbers + masks).
   - It produces a **logit score for each emotion** per sentence.
   - We compare logits to **true labels** (0 or 1 per emotion).
   - The model **adjusts its weights** to reduce the difference (loss).

2. **Components:**
   - **Model:** roberta-base + small classification layer
   - **Loss function:** BCEWithLogitsLoss (for multi-label)
   - **Optimizer:** AdamW (updates weights based on gradients)
   - **DataLoader:** Feeds batches of sentences

3. **Training flow:**
   - For each batch:
     1. Clear old gradients
     2. Feed batch to model → get logits
     3. Compute loss (logits vs labels)
     4. Backpropagate → compute gradients
     5. Optimizer updates model weights
   - Repeat for all batches (1 epoch = 1 pass through all data)

In [20]:
import torch
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW  # Correct import for AdamW

# -------------------------------
# Device setup (basically tells us whether we are training on CPU or GPU, since we are on laptops we would be using CPU, generally slower than using GPU)
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

# -------------------------------
# Load the model
# -------------------------------
num_emotions = y_train.shape[1]  # number of emotion labels
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=num_emotions  # Hyperparameter: number of outputs
)
model.to(device)

# -------------------------------
# Loss function
# -------------------------------
criterion = torch.nn.BCEWithLogitsLoss()  # Hyperparameter: loss type for multi-label

# -------------------------------
# Optimizer
# -------------------------------
optimizer = AdamW(model.parameters(), lr=2e-5)  # Hyperparameter: learning rate
# Note: model.parameters() = Parameters (weights) that will be updated

# -------------------------------
# Example Training Loop (1 batch, 1 epoch)
# -------------------------------
model.train()  # set model to training mode
for batch in train_loader:
    # Move batch data to device
    input_ids, attention_mask, labels = [x.to(device) for x in batch]

    # ---------------------------
    # Forward pass
    # ---------------------------
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits  # raw scores for each emotion

    # ---------------------------
    # Compute loss
    # ---------------------------
    loss = criterion(logits, labels)  # compare predictions vs true labels
    loss.backward()                    # compute gradients for parameters

    # ---------------------------
    # Update weights
    # ---------------------------
    optimizer.step()    # optimizer updates model parameters
    optimizer.zero_grad()  # clear gradients for next step

    print("Batch loss:", loss.item())


Training on: cpu


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 528.43it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Batch loss: 0.6974436044692993
Batch loss: 0.6899774670600891
Batch loss: 0.6827057600021362
Batch loss: 0.6745473146438599
Batch loss: 0.6701153516769409
Batch loss: 0.658978283405304
Batch loss: 0.6443678140640259
Batch loss: 0.6361278295516968
Batch loss: 0.6154918670654297
Batch loss: 0.6034543514251709
Batch loss: 0.5577423572540283
Batch loss: 0.5239134430885315
Batch loss: 0.4830314517021179
Batch loss: 0.46327143907546997
Batch loss: 0.44991740584373474
Batch loss: 0.42771679162979126
Batch loss: 0.40496426820755005
Batch loss: 0.40426963567733765
Batch loss: 0.37806758284568787
Batch loss: 0.36386698484420776
Batch loss: 0.35191869735717773
Batch loss: 0.34232157468795776
Batch loss: 0.3281974196434021
Batch loss: 0.3264334499835968
Batch loss: 0.3098929524421692
Batch loss: 0.30066248774528503
Batch loss: 0.3029460608959198
Batch loss: 0.3023114800453186
Batch loss: 0.29810968041419983
Batch loss: 0.29583537578582764
Batch loss: 0.2804650664329529
Batch loss: 0.28246122598648

KeyboardInterrupt: 

## Step 7 — Evaluation & Testing



After fine-tuning the model, we want to see how well it performs on unseen data.

This step is about checking predictions against the ground truth without updating weights.

In [ ]:
from sklearn.metrics import f1_score

# Move evaluation data to device
val_input_ids = val_tokens['input_ids'].to(device)
val_attention_mask = val_tokens['attention_mask'].to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.float).to(device)

# Set model to evaluation mode
model.eval()

with torch.no_grad():  # no gradients needed
    outputs = model(input_ids=val_input_ids, attention_mask=val_attention_mask)
    logits = outputs.logits
    probs = torch.sigmoid(logits)  # convert logits to probabilities

# Apply threshold to get predicted labels
y_pred = (probs > 0.5).int().cpu().numpy()  # convert back to numpy

# Compute F1-score for multi-label classification
f1 = f1_score(y_val, y_pred, average='micro')  # micro-average across all emotions
print("Validation F1-score:", f1)


## Step 8 — Inference on New Sentences

After the model is trained and validated, you can predict emotions for new text.

### Steps:

1. Tokenize the new sentences just like training data.

2. Move tensors to the same device as your model (CPU/GPU).

3. Forward pass through the model.

4. Convert logits to probabilities using sigmoid (for multi-label).

5. Apply a threshold (e.g., 0.5) to determine predicted emotions.

---

### Example new sentences

```python
new_texts = [
    "I just got a promotion at work!",
    "I'm feeling nervous and scared about the exam tomorrow.",
    "Why did they do that to me? I'm so angry!"
]
```
---


In [ ]:
# Example sentences to predict emotions
new_texts = [
    "I just got a promotion at work!",
    "I'm feeling nervous and scared about the exam tomorrow.",
    "Why did they do that to me? I'm so angry!"
]

# Tokenize new sentences
new_tokens = tokenizer(
    new_texts,
    padding='max_length',   # pad all sentences to same length
    truncation=True,        # truncate long sentences
    max_length=64,
    return_tensors='pt'     # return PyTorch tensors
)

# Move tensors to device (CPU/GPU)
input_ids = new_tokens['input_ids'].to(device)
attention_mask = new_tokens['attention_mask'].to(device)

# Set model to evaluation mode
model.eval()

with torch.no_grad():  # no gradients needed
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    probs = torch.sigmoid(logits)  # convert logits to probabilities

# Convert probabilities to 0/1 predictions
y_pred = (probs > 0.5).int().cpu().numpy()

# Map predictions to emotion labels
emotion_labels = list(emotion_cols)  # same as dataset columns
for i, text in enumerate(new_texts):
    predicted_emotions = [emotion_labels[j] for j, val in enumerate(y_pred[i]) if val == 1]
    print(f"Text: {text}")
    print(f"Predicted emotions: {predicted_emotions}\n")

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# --- Get validation probabilities (you already have probs/logits logic, but we'll recompute cleanly) ---
model.eval()
with torch.no_grad():
    val_input_ids = val_tokens["input_ids"].to(device)
    val_attention_mask = val_tokens["attention_mask"].to(device)

    logits = model(input_ids=val_input_ids, attention_mask=val_attention_mask).logits
    probs = torch.sigmoid(logits).cpu().numpy()

y_true = y_val.astype(int)

# --- Sweep thresholds ---
thresholds = np.arange(0.05, 0.96, 0.05)
best_t = 0.5
best_micro = -1

for t in thresholds:
    y_pred = (probs >= t).astype(int)
    micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    if micro > best_micro:
        best_micro = micro
        best_t = t

print("Best global threshold:", best_t)
print("Best micro-F1:", round(best_micro, 4))